# Exploratory Data Analysis: Root Cross-Section Segmentation

Exploratory dataset analysis focusing on biological specimen groupings, cross-section subtypes, and leakage auditing for nnU-Net v2 conversion.

## 1. Environment Setup & Configuration

In [14]:
from pathlib import Path
import pandas as pd
from hydra import compose, initialize
import rootutils
from PIL import Image

# Setup project root and pythonpath
rootutils.setup_root(Path.cwd(), indicator="pyproject.toml", pythonpath=True)

import src.utils.resolvers

# Load Hydra configuration
with initialize(version_base="1.3", config_path="../cfg"):
    cfg = compose(config_name="config")

# Import domain utilities
from src.utils.coco import load_coco, extract_group
from src.utils.convert import make_grouped_partition

## 2. Dataset Ingestion & Specimen Metadata Extraction

Parse Roboflow COCO annotations across all configured splits and extract biological specimen identifiers (`group`) and section subtypes (`subgroup`).

In [15]:
# Read split configurations from Hydra
dataset_dir = Path(cfg.paths.dataset)
split_cfg = cfg.export.split

group_pattern = split_cfg.group_pattern
subgroup_pattern = split_cfg.get("subgroup_pattern")

records = []
for split_name in cfg.export.roboflow_splits:
    split_dir = dataset_dir / split_name
    if not split_dir.is_dir():
        continue

    coco = load_coco(split_dir)
    for img in coco["images"]:
        file_name = img["file_name"]

        # Extract specimen identifier and section subtype
        group = extract_group(file_name, group_pattern)
        subgroup = extract_group(file_name, subgroup_pattern) if subgroup_pattern else None

        records.append({
            "key": f"{split_name}/{file_name}",
            "roboflow_split": split_name,
            "file_name": file_name,
            "group": group,
            "subgroup": subgroup,
            "width": img["width"],
            "height": img["height"],
        })

df_cases = pd.DataFrame(records)
unique_groups = sorted(df_cases["group"].unique())
unique_subgroups = sorted(df_cases["subgroup"].dropna().unique())

print(f"Total images: {len(df_cases)}")
print(f"Unique specimen groups ({len(unique_groups)}): {unique_groups}")
if subgroup_pattern:
    print(f"Unique subgroups ({len(unique_subgroups)}): {unique_subgroups}")

Total images: 150
Unique specimen groups (20): ['1501', '1502', '1503', '1504', '1505', '1601', '1602', '1603', '1604', '1605', '1701', '1702', '1703', '1704', '1705', '1801', '1802', '1803', '1804', '1805']
Unique subgroups (2): ['1', '2']


## 3. Specimen Distribution & Subgroup Breakdown

Inspect the number of cross-section images per biological specimen and examine how subtypes are distributed across groups.

In [16]:
# Summary of cross-section images per biological specimen
print("--- Images per Specimen Summary ---")
print(df_cases.groupby("group").size().describe())

# Distribution of section subtypes across biological specimens
if subgroup_pattern:
    print("\n--- Subgroup Distribution by Specimen ---")
    subgroup_dist = df_cases.groupby(["group", "subgroup"]).size().unstack(fill_value=0)
    display(subgroup_dist)

--- Images per Specimen Summary ---
count    20.000000
mean      7.500000
std       1.791794
min       4.000000
25%       7.750000
50%       8.000000
75%       8.000000
max      12.000000
dtype: float64

--- Subgroup Distribution by Specimen ---


subgroup,1,2
group,,
1501,8,4
1502,0,4
1503,0,4
1504,4,4
1505,4,4
1601,4,4
1602,4,4
1603,4,4
1604,4,4


## 4. Roboflow Split Leakage Audit

Check whether Roboflow's default assignment split slices of the same specimen across `train`, `valid`, and `test` subsets, causing data leakage.

In [17]:
# Audit specimen overlap across Roboflow subsets
leakage_by_group = df_cases.groupby("group")["roboflow_split"].nunique()
leaked_groups = leakage_by_group[leakage_by_group > 1]

if len(leaked_groups) > 0:
    print(f"[WARNING] Detected {len(leaked_groups)} specimen(s) split across Roboflow subsets (data leakage):")
    leaked_df = (
        df_cases[df_cases["group"].isin(leaked_groups.index)]
        .groupby(["group", "roboflow_split"])
        .size()
        .unstack(fill_value=0)
    )
    display(leaked_df)
else:
    print("[OK] No specimen overlap detected across Roboflow splits.")

# Subgroup balance across original Roboflow splits
if subgroup_pattern:
    print("\n--- Subgroup Distribution Across Roboflow Splits ---")
    display(df_cases.groupby(["roboflow_split", "subgroup"]).size().unstack(fill_value=0))

[WARNING] Detected 20 specimen(s) split across Roboflow subsets (data leakage):


roboflow_split,test,train,valid
group,,,
1501,2,8,2
1502,0,3,1
1503,0,3,1
1504,1,7,0
1505,0,6,2
1601,0,5,3
1602,1,4,3
1603,1,5,2
1604,1,7,0



--- Subgroup Distribution Across Roboflow Splits ---


subgroup,1,2
roboflow_split,,
test,9,6
train,47,58
valid,14,16


## 5. nnU-Net Grouped Partitioning Simulation

Simulate specimen-grouped partitioning to produce a held-out test set and balanced 5-fold cross-validation folds without data leakage.

In [18]:
# Prepare key-to-group mappings for partition search
group_of = dict(zip(df_cases["key"], df_cases["group"]))
subgroup_of = dict(zip(df_cases["key"], df_cases["subgroup"])) if subgroup_pattern else None

# Search for optimal test set and cross-validation folds
test_keys, fold_of = make_grouped_partition(
    group_of=group_of,
    subgroup_of=subgroup_of,
    test_frac=float(split_cfg.test_fraction),
    n_folds=int(split_cfg.n_folds),
    seed=int(split_cfg.seed),
)

# Assign partition assignments back to DataFrame
df_cases["nnunet_partition"] = df_cases["key"].apply(
    lambda k: "test" if k in test_keys else f"fold_{fold_of[k]}"
)

# Aggregate images and group counts per partition
partition_summary = df_cases.groupby(["nnunet_partition", "subgroup"]).size().unstack(fill_value=0)
partition_summary["total_images"] = partition_summary.sum(axis=1)
partition_summary["unique_groups"] = df_cases.groupby("nnunet_partition")["group"].nunique()

print("--- nnU-Net Grouped Partition Summary ---")
display(partition_summary)

--- nnU-Net Grouped Partition Summary ---


subgroup,1,2,total_images,unique_groups
nnunet_partition,,,,
fold_0,12,16,28,4
fold_1,12,12,24,3
fold_2,12,12,24,3
fold_3,12,12,24,3
fold_4,11,16,27,4
test,11,12,23,3


## 6. Partition Verification & Zero-Leakage Guarantee

Assert that each biological specimen is strictly isolated in exactly one partition (either the test set or a single training fold).

In [19]:
# Assert zero specimen leakage across nnU-Net partitions
partition_counts_per_group = df_cases.groupby("group")["nnunet_partition"].nunique()
multi_partition_groups = partition_counts_per_group[partition_counts_per_group > 1]

assert len(multi_partition_groups) == 0, (
    f"Leakage detected! Specimen groups assigned to multiple partitions: {list(multi_partition_groups.index)}"
)

print("[PASSED] Verification successful: every specimen is strictly isolated in a single partition.")

[PASSED] Verification successful: every specimen is strictly isolated in a single partition.
